# 02: Titanic OpenML Reference Workflow

**Purpose:** Reference workflow demonstrating dataset difference and separate modelling on the larger OpenML Titanic dataset.

### Critical Rules:
1. Do not mix OpenML rows with Kaggle rows.
2. Do not train one model using both datasets together.
3. OpenML dataset commonly contains around 1,309 records with additional columns like `boat`, `body`, and `home.dest`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

sns.set_theme(style="whitegrid")


## 1. Loading OpenML Titanic Dataset


In [ ]:
url = "https://www.openml.org/data/get_csv/16826755/phpMYEkMl"
try:
    df = pd.read_csv(url)
    print("✓ Loaded Titanic dataset from OpenML URL!")
except Exception as e:
    print("Failed loading URL. Loading secondary source...")
    df = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")

df.columns = df.columns.str.strip().str.lower()
df = df.replace("?", np.nan)

for num_col in ["age", "fare"]:
    if num_col in df.columns:
        df[num_col] = pd.to_numeric(df[num_col], errors="coerce")

print("OpenML dataset shape:", df.shape)
print(df.info())


## 2. Preprocessing & Reference Modeling
Train model solely on OpenML data as a reference comparison.


In [ ]:
feat_cols = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
df["age"] = df["age"].fillna(df["age"].median())
df["fare"] = df["fare"].fillna(df["fare"].median())
df["embarked"] = df["embarked"].fillna("S")

df["survived"] = df["survived"].astype(int)

X = df[feat_cols]
y = df["survived"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

num_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", drop="first"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", num_transformer, ["age", "sibsp", "parch", "fare"]),
    ("cat", cat_transformer, ["pclass", "sex", "embarked"])
])

model = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, solver="liblinear", random_state=42))
])

model.fit(X_train, y_train)
acc = model.score(X_val, y_val)
print(f"OpenML Baseline Logistic Regression Accuracy: {acc:.4f}")
